<a href="https://colab.research.google.com/github/nithin9000/MedMistral/blob/main/MedMistral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
!pip install -U bitsandbytes
!pip install transformers
!pip install -U peft
!pip install -U accelerate
!pip install -U trlno i
!pip install datasets
!pip install sentencepiece

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,HfArgumentParser,TrainingArguments,pipeline, logging
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
import os,torch
from datasets import load_dataset
from trl import SFTTrainer
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pandas as pd
from datasets import Dataset
import re

In [ ]:
tokenizer = Auto

In [3]:
from google.colab import userdata
secret_hf = userdata.get('HUGGINGFACE_TOKEN')
!huggingface-cli login --token $secret_hf

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `nithin` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `nithin`


In [ ]:
from datasets import load_dataset

ds = load_dataset("medalpaca/medical_meadow_medical_flashcards")

In [4]:
base_model = "mistralai/Mistral-7B-v0.1"
new_model = "MedMistral"



In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.bfloat16,
    bnb_4bit_use_double_quant = False
)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config = bnb_config,
    load_in_4bit = True,
    torch_dtype = torch.bfloat16,
    device_map = "auto",
    trust_remote_code = True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1
model.gradient_checkpointing_enable()

tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_eos_token = True
tokenizer.bos_token,tokenizer.eos_token

In [ ]:
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj"]

)

model = get_peft_model(model, peft_config)

In [ ]:
#Hyperparameters
training_arguments = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    optim = "paged_adamw_32bit",
    save_steps = 50,
    logging_steps = 25,
    learning_rate = 2e-4,
    weight_decay = 0.001,
    fp16 = True,
    bf16 = False,
    max_grad_norm = 0.3,
    max_steps = -1,
    warmup_ratio = 0.03,
    grouped_by_length = True,
    lr_scheduler_type = "constant"
)

In [ ]:
#SFT Params
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    peft_config = peft_config,
    max_seq_length = None,
    dataset_text_field = "text",
    tokenizer = tokenizer,
    args = training_arguments,
    packing = False,
)

In [ ]:
trainer.train()

In [ ]:
trainer.model.save_pretrained(new_model)
model.config.use_cache = True
model.eval()